# Reddit data access (custom public JSON wrapper)

Il seguente notebook descrive come accedere ai dati di Reddit tramite un wrapper JSON pubblico personalizzato. Questo wrapper consente di ottenere facilmente i dati dei subreddit, dei post e dei commenti senza dover utilizzare l'API ufficiale di Reddit, che richiede autenticazione e può essere complessa da configurare.

## Inizializzazione

In [35]:
import networkx as nx
import time
import requests
from typing import Any, Dict, List, Optional

BASE_URL = "https://reddit.com"
USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"


## Profilo utente
Di seguito è riportata una funzione per ottenere le informazioni di un utente Reddit. La funzione accetta un parametro "where" che specifica quali informazioni si desidera ottenere.

In [27]:
# Funzione per ottenere le informazioni di un utente
# La funzione accetta un parametro "where" che specifica quali informazioni si desidera ottenere:
# - "about": informazioni di base (default)
# - "overview": panoramica combinata di post e commenti
# - "submitted": solo i post inviati
# - "comments": solo i commenti

def get_user_info(username, where="about"):
    url = f"{BASE_URL}/user/{username}/{where}.json"
    headers = {"User-Agent": USER_AGENT}
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json()
    else:
        return None

Esempi:

In [28]:
get_user_info("ciobbernaut")

{'kind': 't2',
 'data': {'is_employee': False,
  'is_friend': False,
  'subreddit': {'default_set': False,
   'user_is_contributor': None,
   'banner_img': '',
   'allowed_media_in_comments': [],
   'user_is_banned': None,
   'free_form_reports': True,
   'community_icon': None,
   'show_media': True,
   'icon_color': '#FF99AA',
   'user_is_muted': None,
   'display_name': 'u_ciobbernaut',
   'header_img': None,
   'title': '',
   'previous_names': [],
   'over_18': False,
   'icon_size': [256, 256],
   'primary_color': '',
   'icon_img': 'https://www.redditstatic.com/avatars/defaults/v2/avatar_default_0.png',
   'description': '',
   'submit_link_label': '',
   'header_size': None,
   'restrict_posting': True,
   'restrict_commenting': False,
   'subscribers': 0,
   'submit_text_label': '',
   'is_default_icon': True,
   'link_flair_position': '',
   'display_name_prefixed': 'u/ciobbernaut',
   'key_color': '',
   'name': 't5_h979ng',
   'is_default_banner': True,
   'url': '/user/cio

In [29]:
get_user_info("ciobbernaut", where="submitted")

{'kind': 'Listing',
 'data': {'after': None,
  'dist': 1,
  'modhash': '',
  'geo_filter': '',
  'children': [{'kind': 't3',
    'data': {'approved_at_utc': None,
     'subreddit': 'u_ciobbernaut',
     'selftext': 'smm',
     'author_fullname': 't2_2bihu5bei8',
     'saved': False,
     'mod_reason_title': None,
     'gilded': 0,
     'clicked': False,
     'title': 'Prova',
     'link_flair_richtext': [],
     'subreddit_name_prefixed': 'u/ciobbernaut',
     'hidden': False,
     'pwls': None,
     'link_flair_css_class': None,
     'downs': 0,
     'thumbnail_height': None,
     'top_awarded_type': None,
     'hide_score': False,
     'name': 't3_1sey7u9',
     'quarantine': False,
     'link_flair_text_color': 'dark',
     'upvote_ratio': 1.0,
     'author_flair_background_color': None,
     'subreddit_type': 'user',
     'ups': 1,
     'total_awards_received': 0,
     'media_embed': {},
     'thumbnail_width': None,
     'author_flair_template_id': None,
     'is_original_content'

# Ottenere n post di un subreddit

Di seguito è riportata una funzione per ottenere i post di un subreddit specifico. La funzione supporta la paginazione e gestisce i limiti di richiesta per evitare di essere bloccati dal server.

In [36]:
def _sleep_if_rate_limited(headers: Dict[str, str]) -> None:
    """
    Verifica gli header e dorme se necessario.
    Supporta: Retry-After, x-ratelimit-remaining e x-ratelimit-reset (se presenti).
    """
    retry_after = headers.get("Retry-After")
    if retry_after:
        try:
            secs = int(float(retry_after))
            time.sleep(secs + 1)
            return
        except ValueError:
            pass

    remaining = headers.get("x-ratelimit-remaining")
    reset = headers.get("x-ratelimit-reset")
    if remaining is not None:
        try:
            rem_val = float(remaining)
            if rem_val <= 0 and reset is not None:
                secs = int(float(reset))
                time.sleep(secs + 1)
        except ValueError:
            pass

def get_subreddit_posts(subreddit: str, limit: int = 10, max_per_request: int = 100
                       ) -> Dict[str, Any]:
    """
    Recupera fino a `limit` post dal feed `new` di un subreddit usando paginazione se necessario.
    Restituisce un dict con:
      - 'posts': lista di post (ogni post è il dict `data` dell'elemento)
      - 'rate_limit_headers': ultimi header utili al rate limiting (se presenti)
    """
    headers = {"User-Agent": USER_AGENT}
    posts: List[Dict[str, Any]] = []
    after: Optional[str] = None
    remaining_to_fetch = max(0, limit)
    last_rate_headers: Dict[str, str] = {}

    while remaining_to_fetch > 0:
        per_request = min(remaining_to_fetch, max_per_request)
        params = {"limit": per_request}
        if after:
            params["after"] = after

        try:
            resp = requests.get(f"{BASE_URL}/r/{subreddit}/new.json", headers=headers, params=params)
        except requests.RequestException:
            break

        # salva headers utili
        # normalizzare le key in lower-case per sicurezza
        resp_headers = {k.lower(): v for k, v in resp.headers.items()}
        # conserva solo quelli utili
        for key in ("x-ratelimit-remaining", "x-ratelimit-reset", "retry-after"):
            if key in resp_headers:
                last_rate_headers[key] = resp_headers[key]

        if resp.status_code == 429:
            # troppi richieste: attendi e riprova
            _sleep_if_rate_limited(resp.headers)
            continue

        if resp.status_code != 200:
            # interrompi in caso di errore non recuperabile
            break

        try:
            payload = resp.json()
        except ValueError:
            break

        children = payload.get("data", {}).get("children", [])
        for child in children:
            # alcune volte ogni elemento è { "kind": .., "data": {...} }
            data = child.get("data", child) if isinstance(child, dict) else child
            posts.append(data)
            remaining_to_fetch -= 1
            if remaining_to_fetch <= 0:
                break

        after = payload.get("data", {}).get("after")
        # se non ci sono altre pagine esci
        if not after:
            break

        # Controlla header rate limit: se esaurito, attendi
        _sleep_if_rate_limited(resp.headers)

    return {"posts": posts, "rate_limit_headers": last_rate_headers}


In [49]:
resp = get_subreddit_posts("italy", limit=5)

In [50]:
resp["posts"][2]

{'approved_at_utc': None,
 'subreddit': 'italy',
 'selftext': 'Buongiorno amici sportivi e non sportivi.  \n\n\n* Prima semifinale del singolare del master 1000 di Montecarlo tra Sinner e Zverev, sul campo centrale i due campioni si affronteranno sabato non prima delle 13:30. Sarà possibile vedere la diretta su Sky, mentre TV8 offrirà la differita alle ore 14:30.  \n\n* Torna anche la serie A, queste le partite in programma:  \n\n* Roma- Pisa  \n* Torino - Verona  \n* Cagliari - Cremonese  \n* Milan - Udinese  \n* Atalanta - Juventus  \n* Genoa - Sassuolo  \n* Parma - Napoli  \n* Bologna - Lecce  \n* Como - Inter  \n* Fiorentina - Lazio\n\n\nSe avete altri avvenimenti sportivi di cui volete parlare, fatemelo sapere nei commenti.',
 'author_fullname': 't2_kluup',
 'saved': False,
 'mod_reason_title': None,
 'gilded': 0,
 'clicked': False,
 'title': 'Bar sport di r/italy',
 'link_flair_richtext': [],
 'subreddit_name_prefixed': 'r/italy',
 'hidden': False,
 'pwls': 6,
 'link_flair_css_cl

# Ottenere i commenti di un post (a i primi due livelli)

Di seguito è riportata una funzione per ottenere n commenti di un post specifico. La funzione accetta l'ID del post e restituisce una lista di commenti.

In [73]:
def get_post_comments(post_id: str, limit: int = 10) -> List[Dict[str, Any]]:
    url = f"{BASE_URL}/comments/{post_id}.json"
    headers = {"User-Agent": USER_AGENT}
    params = {"limit": limit}
    response = requests.get(url, headers=headers, params=params)
    if response.status_code == 200:
        try:
            data = response.json()
            # I commenti sono nella seconda parte della risposta
            comments = data[1].get("data", {}).get("children", [])
            return [comment.get("data", {}) for comment in comments if comment.get("kind") == "t1"]
        except ValueError:
            return []
    else:
        return []

In [75]:
get_post_comments(resp["posts"][3]["id"], limit=10)

[{'subreddit_id': 't5_2qkhk',
  'approved_at_utc': None,
  'author_is_blocked': False,
  'comment_type': None,
  'awarders': [],
  'mod_reason_by': None,
  'banned_by': None,
  'author_flair_type': 'richtext',
  'total_awards_received': 0,
  'subreddit': 'italy',
  'author_flair_template_id': 'd14a7942-af3f-11e7-aa14-0e240cdc8de0',
  'likes': None,
  'replies': {'kind': 'Listing',
   'data': {'after': None,
    'dist': None,
    'modhash': '',
    'geo_filter': '',
    'children': [{'kind': 'more',
      'data': {'count': 3,
       'name': 't1_ofk3ade',
       'id': 'ofk3ade',
       'parent_id': 't1_ofjqhom',
       'depth': 1,
       'children': ['ofk3ade']}}],
    'before': None}},
  'user_reports': [],
  'saved': False,
  'id': 'ofjqhom',
  'banned_at_utc': None,
  'mod_reason_title': None,
  'gilded': 0,
  'archived': False,
  'collapsed_reason_code': None,
  'no_follow': False,
  'author': 'An_Oxygen_Consumer',
  'can_mod_post': False,
  'created_utc': 1775904632.0,
  'send_repli

# Esempio di costruzione di un grafo con gli utenti che hanno commentato un post

In [103]:
posts = get_subreddit_posts("italy", limit=10)["posts"]
G = nx.Graph()

for post in posts:
    comments = get_post_comments(post["id"], limit=10)
    G.add_node(post["id"], bipartite=1)
    for comment in comments:
        author = comment.get("author")
        if author:
            G.add_node(author, bipartite=0)
            # Aggiungi un edge tra l'autore del commento e il post (o tra autori di commenti se vuoi)
            G.add_edge(author, post["id"])

In [104]:
nx.write_gexf(G,path = './data/graph.gexf')

# Visualizzazione in Gephi

<img src="./data/grafo.png">
